# CHB-MIT Corr-DCRNN — Correlation Graph Visualization

Visualizes the per-clip correlation adjacency matrix generated by
`CHBMITDatasetHDF5._get_indiv_graph()` (paper: Tang et al., ICLR 2022).

**Outputs (saved to `./chbmit_outputs/`):**
1. `corr_heatmap_<patient>_label<N>.png` — signed + absolute correlation heatmap with top-k overlay
2. `corr_graph_<patient>_label<N>.png` — full weighted graph vs sparse top-k graph
3. `corr_comparison_<patient>.png` — seizure vs. background side-by-side comparison

**Pipeline recap:**
```
HDF5 slice (C, 12×256) → resample (C, 12×200) → reshape (12, C, 200)
  → flatten (C, 12×200) → gram / (||row_i|| × ||row_j||) → corr_mat (C, C)
  → abs → top-k=3 sparsification → adj_topk (C, C) directed
```

In [ ]:
import sys
import os
from pathlib import Path

# Add repo root to path (notebook lives in graph_viz/)
REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'graph_viz'))

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import networkx as nx
from scipy.stats import rankdata

# Reuse existing TUSZ graph viz utilities
from graph_viz_utils import draw_graph_weighted_edge

# CHB-MIT pipeline
from data.data_utils import keep_topk
from data.dataloader_chbmit_hdf5 import load_dataset_chbmit_hdf5

# Inline output for notebook
%matplotlib inline
matplotlib.rcParams['figure.dpi'] = 120

print(f"Repo root: {REPO_ROOT}")

## 1. Configuration

Edit the paths and parameters below before running.

In [ ]:
# ── EDIT THESE ──────────────────────────────────────────────────────────────
HDF5_DIR    = "/vessl/data/chbmit_hdf5"   # root dir with CHB-MIT HDF5 files
SUMMARY_DIR = None                          # optional: dir with *-summary.txt
SAMPLE_IDX  = None                          # None = auto-select (prefer seizure)

WIN_LEN    = 12    # clip length in seconds (paper: 12)
ORIG_FS    = 256   # native HDF5 sampling rate
TARGET_FS  = 200   # resampling target (paper)
FFT_FEAT   = 100   # FFT bins (unused here but needed for dataset init)
TOP_K      = 3     # top-k neighbours per node (paper: τ=3)
# ────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR = Path("./chbmit_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Outputs → {OUTPUT_DIR.resolve()}")

## 2. Load Dataset (train split, no standardization)

In [ ]:
import logging
logging.basicConfig(level=logging.WARNING)   # suppress INFO-level build logs

dataloaders, datasets, scaler = load_dataset_chbmit_hdf5(
    hdf5_dir=HDF5_DIR,
    summary_dir=SUMMARY_DIR,
    win_len=WIN_LEN,
    orig_fs=ORIG_FS,
    fs=TARGET_FS,
    fft_features=FFT_FEAT,
    graph_type="individual",
    top_k=TOP_K,
    standardize=False,      # no normalization needed for visualization
    num_workers=0,
    undersample_train=False,
    inspect_first_file=True,
)

ds = datasets["train"]
C  = ds.num_nodes
print(f"\nDataset: {len(ds)} windows | {C} channels | seq_len={ds.seq_len}")
print(f"Seizure windows: {sum(e.label for e in ds.index)}")
print(f"Channel names from HDF5: {ds.channels}")

## 3. Select Sample

In [ ]:
if SAMPLE_IDX is None:
    sz_indices = [i for i, e in enumerate(ds.index) if e.label == 1]
    if sz_indices:
        idx = sz_indices[0]
        print(f"Auto-selected SEIZURE sample: idx={idx}")
    else:
        idx = 0
        print("No seizure samples found — using idx=0 (background)")
else:
    idx = SAMPLE_IDX

entry = ds.index[idx]
print(f"patient={entry.patient}  start={entry.start}  "
      f"label={entry.label} ({'seizure' if entry.label else 'background'})")

## 4. Compute Correlation Matrix & Top-k Adjacency

Replicates `_get_indiv_graph()` step-by-step so we can inspect the intermediate
correlation matrix before and after top-k sparsification.

In [ ]:
# ── Load & resample raw EEG window ─────────────────────────────────────────
raw = ds._load_window(entry)             # (C, T_resampled) at target_fs
n_steps = ds.seq_len
step_s  = ds.step_samples               # = target_fs (200)

eeg_clip = raw.reshape(C, n_steps, step_s).transpose(1, 0, 2).copy()
print(f"raw window  : {raw.shape}  = (C={C}, T={raw.shape[1]}) @{TARGET_FS}Hz")
print(f"eeg_clip    : {eeg_clip.shape}  = (seq_len={n_steps}, C={C}, step_samples={step_s})")

# ── Gram-matrix cross-correlation (mirrors _get_indiv_graph exactly) ────────
flat = eeg_clip.transpose(1, 0, 2).reshape(C, -1).astype(np.float64)   # (C, T*S)
gram = flat @ flat.T                                                      # (C, C)

norms     = np.sqrt(np.maximum(np.diag(gram), 0.0))                     # (C,)
norm_prod = np.outer(norms, norms)                                        # (C, C)
norm_prod[norm_prod < 1e-12] = 1.0

corr_mat  = (gram / norm_prod).astype(np.float32)                        # (C, C) ∈ [-1, 1]

# ── Absolute + top-k ────────────────────────────────────────────────────────
adj_abs  = np.abs(corr_mat)
np.fill_diagonal(adj_abs, 1.0)
adj_topk = keep_topk(adj_abs, top_k=TOP_K, directed=True)               # (C, C) sparse

off_diag = corr_mat[~np.eye(C, dtype=bool)]
nnz      = int(np.count_nonzero(adj_topk)) - C   # exclude self-edges

print(f"\ncorr_mat stats: min={corr_mat.min():.3f}  max={corr_mat.max():.3f}  "
      f"off-diag mean={off_diag.mean():.3f}  std={off_diag.std():.3f}")
print(f"adj_topk: nnz_edges={nnz}  (top-{TOP_K} per node, directed)")

## 5. Channel Name Helper

In [ ]:
def clean_ch_name(raw_name: str) -> str:
    """Strip 'EEG ' prefix and '-REF'/'-LE' suffix for compact axis labels."""
    name = raw_name.strip()
    for prefix in ("EEG ", "eeg "):
        if name.startswith(prefix):
            name = name[len(prefix):]
    for suffix in ("-REF", "-LE", "-ref", "-le"):
        if name.endswith(suffix):
            name = name[:-len(suffix)]
    return name

if ds.channels:
    ch_names = [clean_ch_name(c) for c in ds.channels]
else:
    ch_names = [f"Ch{i:02d}" for i in range(C)]

# node_id_dict format expected by draw_graph_weighted_edge: {name: index}
node_id_dict = {name: i for i, name in enumerate(ch_names)}

print(f"Channel labels ({C}): {ch_names}")

## 6. Layout Helper

CHB-MIT channels are patient-specific and don't always map to standard 10-20
electrode positions, so we use a **circular layout** as the default.

> If your recordings have exactly the standard 19 channels (FP1/FP2/F3/…/O2),
> you can substitute `get_spectral_graph_positions()` from `graph_viz_utils.py`.

In [ ]:
def get_circular_positions(n_nodes: int) -> dict:
    """Circular layout; maps node index → (x, y). Works for any channel count."""
    G_tmp = nx.cycle_graph(n_nodes)
    return nx.circular_layout(G_tmp)

pos_circ = get_circular_positions(C)
labels   = {i: ch_names[i] for i in range(C)}

# ── Optional: approximate 10-20 scalp positions ────────────────────────────
# Covers common CHB-MIT channel names (cleaned, no EEG prefix / -REF suffix)
_POS_10_20 = {
    "FP1": (-0.30,  0.90), "FP2": ( 0.30,  0.90),
    "F7":  (-0.70,  0.50), "F3":  (-0.40,  0.50), "FZ": (0.0,  0.50),
    "F4":  ( 0.40,  0.50), "F8":  ( 0.70,  0.50),
    "T3":  (-0.95,  0.00), "C3":  (-0.50,  0.00), "CZ": (0.0,  0.00),
    "C4":  ( 0.50,  0.00), "T4":  ( 0.95,  0.00),
    "T5":  (-0.70, -0.50), "P3":  (-0.40, -0.50), "PZ": (0.0, -0.50),
    "P4":  ( 0.40, -0.50), "T6":  ( 0.70, -0.50),
    "O1":  (-0.30, -0.90), "O2":  ( 0.30, -0.90),
    # A1/A2 ear references
    "A1":  (-1.05,  0.00), "A2":  ( 1.05,  0.00),
}

def get_scalp_positions(ch_names: list) -> dict:
    """Return 10-20 scalp positions where available; fall back to circular."""
    circ = get_circular_positions(len(ch_names))
    return {
        i: _POS_10_20.get(name.upper(), circ[i])
        for i, name in enumerate(ch_names)
    }

pos_scalp = get_scalp_positions(ch_names)
n_mapped  = sum(1 for n in ch_names if n.upper() in _POS_10_20)
print(f"{n_mapped}/{C} channels mapped to 10-20 scalp positions.")
print("(Using pos_scalp for anatomical layout, pos_circ as fallback.)")

## 7. Figure 1 — Correlation Heatmap

Left: **signed** cross-correlation ∈ [-1, 1]  
Right: **absolute** cross-correlation ∈ [0, 1] with top-k selected edges highlighted (red border)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))
sample_title = (f"patient={entry.patient}  "
                f"start={entry.start//ORIG_FS:.0f}s  "
                f"label={'seizure' if entry.label else 'background'}")

# ── Left: signed correlation ─────────────────────────────────────────────────
ax = axes[0]
im = ax.imshow(corr_mat, vmin=-1, vmax=1, cmap="RdBu_r", aspect="auto")
ax.set_xticks(range(C))
ax.set_xticklabels(ch_names, rotation=90, fontsize=7 if C > 20 else 9)
ax.set_yticks(range(C))
ax.set_yticklabels(ch_names, fontsize=7 if C > 20 else 9)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title(f"Signed Cross-Correlation\n{sample_title}", fontsize=11)

# ── Right: absolute correlation + top-k overlay ──────────────────────────────
ax = axes[1]
im2 = ax.imshow(adj_abs, vmin=0, vmax=1, cmap="Blues", aspect="auto")
ax.set_xticks(range(C))
ax.set_xticklabels(ch_names, rotation=90, fontsize=7 if C > 20 else 9)
ax.set_yticks(range(C))
ax.set_yticklabels(ch_names, fontsize=7 if C > 20 else 9)
plt.colorbar(im2, ax=ax, fraction=0.046, pad=0.04)

# Highlight selected top-k cells with a red border
for i in range(C):
    for j in range(C):
        if i != j and adj_topk[i, j] > 0:
            ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1,
                         fill=False, edgecolor='red', linewidth=1.0))
ax.set_title(f"|Correlation|  (red box = top-{TOP_K} edge)\n{sample_title}", fontsize=11)

plt.suptitle("CHB-MIT EEG Correlation Matrix", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()

save_path = OUTPUT_DIR / f"corr_heatmap_{entry.patient}_label{entry.label}.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
print(f"Saved → {save_path}")
plt.show()

## 8. Figure 2 — Graph Visualization

Left: **Full weighted** correlation graph (all channel pairs)  
Right: **Sparse top-k** graph used as DCRNN adjacency input

Both use the same `draw_graph_weighted_edge` signature from `graph_viz_utils.py`;
here we call it with `save_dir=None` and embed into subplots manually.

In [ ]:
def build_digraph(adj_mx: np.ndarray, ch_names: list) -> nx.DiGraph:
    """Build a weighted DiGraph from an adjacency matrix."""
    G = nx.DiGraph()
    for i in range(adj_mx.shape[0]):
        G.add_node(i)
    for i in range(adj_mx.shape[0]):
        for j in range(adj_mx.shape[1]):
            if i != j and adj_mx[i, j] > 0:
                G.add_edge(i, j, weight=float(adj_mx[i, j]))
    return G


def draw_chbmit_graph(adj_mx, pos, ch_names, ax,
                      title="", node_color="steelblue",
                      edge_cmap=plt.cm.Greys, edge_width=2.0):
    """
    Axes-aware wrapper around nx.draw_networkx.
    Replicates the coloring logic of graph_viz_utils.draw_graph_weighted_edge.
    """
    G = build_digraph(adj_mx, ch_names)
    labels = {i: ch_names[i] for i in range(len(ch_names))}

    if G.number_of_edges() == 0:
        nx.draw_networkx_nodes(G, pos, ax=ax,
                               node_color=node_color, node_size=900)
        nx.draw_networkx_labels(G, pos, labels=labels, ax=ax,
                                font_color="white", font_size=7)
        ax.set_title(f"{title}\n(no edges)", fontsize=11)
        ax.axis("off")
        return

    edge_weights = [d["weight"] for _, _, d in G.edges(data=True)]
    # Use rank-transformed weights so colour varies even for small differences
    edge_colors  = rankdata(edge_weights)

    # Build colormap matching graph_viz_utils style
    k = 3
    cmap_arr = edge_cmap(np.linspace(0, 1, (k + 1) * len(edge_weights)))
    cmap_obj = mcolors.ListedColormap(
        cmap_arr[len(edge_weights):-1:(k - 1)])

    nx.draw_networkx(
        G, pos, labels=labels, with_labels=True, ax=ax,
        edge_color=edge_colors, edge_cmap=cmap_obj,
        width=edge_width,
        node_color=node_color, node_size=900,
        font_color="white", font_size=7, font_weight="bold",
        arrows=True, arrowsize=12,
    )
    ax.set_title(title, fontsize=11)
    ax.axis("off")


# ── Draw ──────────────────────────────────────────────────────────────────────
use_scalp = n_mapped >= C // 2   # use scalp positions if ≥50% channels mapped
pos_use   = pos_scalp if use_scalp else pos_circ
layout_lbl = "10-20 scalp" if use_scalp else "circular"

fig, axes = plt.subplots(1, 2, figsize=(22, 10))

draw_chbmit_graph(
    adj_abs, pos_use, ch_names, axes[0],
    title=f"Full Weighted Graph (all {C*(C-1)} directed pairs)\n"
          f"patient={entry.patient}, label={'seizure' if entry.label else 'background'}  [{layout_lbl} layout]",
    node_color="steelblue", edge_cmap=plt.cm.Greys, edge_width=1.2,
)
draw_chbmit_graph(
    adj_topk, pos_use, ch_names, axes[1],
    title=f"Top-{TOP_K} Sparse Graph  (nnz={nnz} edges)\n"
          f"patient={entry.patient}, label={'seizure' if entry.label else 'background'}  [{layout_lbl} layout]",
    node_color="crimson", edge_cmap=plt.cm.Reds, edge_width=3.0,
)

plt.suptitle("CHB-MIT Corr-DCRNN Adjacency Graphs", fontsize=14, fontweight="bold")
plt.tight_layout()

save_path = OUTPUT_DIR / f"corr_graph_{entry.patient}_label{entry.label}.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
print(f"Saved → {save_path}")
plt.show()

## 9. Figure 3 — Standalone via `draw_graph_weighted_edge`

Demonstrates reuse of the original `graph_viz_utils.draw_graph_weighted_edge`
with CHB-MIT data (single figure, saved to PNG).

In [ ]:
save_path_standalone = str(OUTPUT_DIR / f"topk_standalone_{entry.patient}_label{entry.label}.png")

draw_graph_weighted_edge(
    adj_mx     = adj_topk,
    node_id_dict = node_id_dict,
    pos_spec   = pos_use,            # circular or scalp positions
    is_directed = True,
    title       = (f"CHB-MIT Top-{TOP_K} Corr Graph  "
                   f"(patient={entry.patient}, "
                   f"{'seizure' if entry.label else 'background'})"),
    save_dir    = save_path_standalone,
    fig_size    = (14, 10),
    node_color  = "crimson" if entry.label else "steelblue",
    font_size   = 12,
    plot_colorbar = True,
)
print(f"Saved → {save_path_standalone}")

## 10. Figure 4 — Seizure vs. Background Comparison

Side-by-side: one seizure clip vs. one background clip from the same patient (if available).

In [ ]:
def compute_corr_adj(ds, idx, C, n_steps, step_s, top_k):
    """Load one window and return (corr_mat, adj_topk, entry)."""
    entry_ = ds.index[idx]
    raw_   = ds._load_window(entry_)
    clip_  = raw_.reshape(C, n_steps, step_s).transpose(1, 0, 2).copy()
    flat_  = clip_.transpose(1, 0, 2).reshape(C, -1).astype(np.float64)
    gram_  = flat_ @ flat_.T
    norms_ = np.sqrt(np.maximum(np.diag(gram_), 0.0))
    np_    = np.outer(norms_, norms_)
    np_[np_ < 1e-12] = 1.0
    corr_  = (gram_ / np_).astype(np.float32)
    abs_   = np.abs(corr_); np.fill_diagonal(abs_, 1.0)
    topk_  = keep_topk(abs_, top_k=top_k, directed=True)
    return corr_, topk_, entry_


# Find one seizure and one background sample from the same patient if possible
sz_entries   = [(i, e) for i, e in enumerate(ds.index) if e.label == 1]
nosz_entries = [(i, e) for i, e in enumerate(ds.index) if e.label == 0]

if not sz_entries:
    print("No seizure samples in this split — skipping comparison.")
else:
    sz_idx_cmp = sz_entries[0][0]
    sz_pat     = sz_entries[0][1].patient
    # Prefer background from the same patient
    same_pat_nosz = [(i, e) for i, e in nosz_entries if e.patient == sz_pat]
    nosz_idx_cmp  = (same_pat_nosz[0][0] if same_pat_nosz
                     else nosz_entries[0][0] if nosz_entries else None)

    if nosz_idx_cmp is None:
        print("No background samples found — skipping comparison.")
    else:
        cm_sz,   tk_sz,   e_sz   = compute_corr_adj(ds, sz_idx_cmp,   C, n_steps, step_s, TOP_K)
        cm_nosz, tk_nosz, e_nosz = compute_corr_adj(ds, nosz_idx_cmp, C, n_steps, step_s, TOP_K)

        fig, axes = plt.subplots(2, 2, figsize=(22, 18))
        rows = [
            (cm_sz,   tk_sz,   e_sz,   "firebrick",  "Reds",  "Seizure  (label=1)"),
            (cm_nosz, tk_nosz, e_nosz, "steelblue",  "Blues", "Background (label=0)"),
        ]

        for row_i, (cm, tk, ent, nc, cmap_name, row_title) in enumerate(rows):
            ent_title = f"{ent.patient}  start={ent.start//ORIG_FS:.0f}s"

            # Heatmap
            ax = axes[row_i][0]
            im = ax.imshow(cm, vmin=-1, vmax=1, cmap="RdBu_r", aspect="auto")
            ax.set_xticks(range(C))
            ax.set_xticklabels(ch_names, rotation=90, fontsize=7 if C > 20 else 8)
            ax.set_yticks(range(C))
            ax.set_yticklabels(ch_names, fontsize=7 if C > 20 else 8)
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            ax.set_title(f"{row_title} — Signed Correlation\n{ent_title}", fontsize=11)

            # Sparse graph
            ax2 = axes[row_i][1]
            draw_chbmit_graph(
                tk, pos_use, ch_names, ax2,
                title=f"{row_title} — Top-{TOP_K} Graph\n{ent_title}",
                node_color=nc, edge_cmap=getattr(plt.cm, cmap_name), edge_width=3.0,
            )

        plt.suptitle("CHB-MIT: Seizure vs. Background Correlation Graphs",
                     fontsize=14, fontweight="bold")
        plt.tight_layout()

        cmp_pat  = e_sz.patient
        save_path = OUTPUT_DIR / f"corr_comparison_{cmp_pat}.png"
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved → {save_path}")
        plt.show()

## 11. Summary — Saved Files

In [ ]:
saved = sorted(OUTPUT_DIR.glob("*.png"))
print(f"\n{'='*55}")
print(f"  Output directory: {OUTPUT_DIR.resolve()}")
print(f"  {len(saved)} PNG file(s) saved:")
for p in saved:
    sz_kb = p.stat().st_size / 1024
    print(f"    {p.name:<55s}  {sz_kb:6.1f} KB")
print(f"{'='*55}")